In [17]:
import pandas_datareader.data as web
from pandas_datareader.famafrench import get_available_datasets
import statsmodels.api as sm
import yfinance as yf

# 1. 下载 Fama-French 官方三因子数据
df_ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2020-01-01')[0]
df_ff

,Mkt-RF,SMB,HML,RF
Date,,,,
2020-01,-0.11,-3.10,-6.22,0.13
2020-02,-8.15,1.08,-3.82,0.12
2020-03,-13.37,-4.69,-13.83,0.12
2020-04,13.60,2.50,-1.34,0.00
2020-05,5.57,2.40,-5.00,0.01
...,...,...,...,...
2025-07,1.98,0.26,-1.27,0.34
2025-08,1.84,3.87,4.42,0.38
2025-09,3.39,-1.85,-1.05,0.33


In [24]:
# 2. 下载单只股票收益率 (例如 AAPL)
stock_data = yf.download('AAPL', start='2020-01-01', progress=False, auto_adjust=True)['Close']
stock_ret = stock_data.pct_change().dropna() * 100 
stock_ret_monthly = (1 + stock_ret/100).resample('M').prod() - 1
stock_ret_monthly = stock_ret_monthly * 100 # 转回百分数
stock_ret_monthly.index = stock_ret_monthly.index.to_period('M')
stock_ret_monthly

Ticker,AAPL
Date,
2020-01,3.049789
2020-02,-11.470146
2020-03,-6.976166
2020-04,15.537406
2020-05,8.509419
...,...
2025-09,9.688125
2025-10,6.181514
2025-11,3.236426


In [25]:
# 3. 合并数据并计算超额收益 (Return - Risk Free)
df_merge = pd.merge(stock_ret_monthly, df_ff, left_index=True, right_index=True)
df_merge.dropna(inplace=True)
df_merge.shape

(71, 5)

In [26]:
df_merge['Excess_Ret'] = df_merge['AAPL'] - df_merge['RF']

df_merge

,AAPL,Mkt-RF,SMB,HML,RF,Excess_Ret
Date,,,,,,
2020-01,3.049789,-0.11,-3.10,-6.22,0.13,2.919789
2020-02,-11.470146,-8.15,1.08,-3.82,0.12,-11.590146
2020-03,-6.976166,-13.37,-4.69,-13.83,0.12,-7.096166
2020-04,15.537406,13.60,2.50,-1.34,0.00,15.537406
2020-05,8.509419,5.57,2.40,-5.00,0.01,8.499419
...,...,...,...,...,...,...
2025-07,1.169770,1.98,0.26,-1.27,0.34,0.829770
2025-08,11.963886,1.84,3.87,4.42,0.38,11.583886
2025-09,9.688125,3.39,-1.85,-1.05,0.33,9.358125


In [27]:
# 4. 运行标准 Fama-French 时间序列回归
# Excess_Ret = alpha + b1*Mkt-RF + b2*SMB + b3*HML
X = df_merge[['Mkt-RF', 'SMB', 'HML']]
X = sm.add_constant(X)
model = sm.OLS(df_merge['Excess_Ret'], X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Excess_Ret   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.542
Method:                 Least Squares   F-statistic:                     28.63
Date:                Mon, 19 Jan 2026   Prob (F-statistic):           4.97e-12
Time:                        16:30:51   Log-Likelihood:                -219.32
No. Observations:                  71   AIC:                             446.6
Df Residuals:                      67   BIC:                             455.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7698      0.670      1.149      0.2